# 🚕 Yo'lchi AI — Sun'iy Intellekt Modelini O'qitish va Test Qilish
Ushbu Google Colab daftarchasi **7,346 ta real o'zbek taksi xabarlari** asosida o'zimizning shaxsiy neyrotarmoq modelimizni o'qitish va jonli sinab ko'rish uchun mo'ljallangan.

### 📌 Qadamlar:
1. Kutubxonalarni yuklash
2. `taxi_training_dataset.csv` faylini yuklash
3. Modelni Train (o'qitish) qilish
4. Imtihon natijalari (Accuracy, F1-Score) ni ko'rish
5. **Interaktiv test:** Xohlagan matningizni kiritib, sun'iy intellekt javobini ko'rish
6. Tayyor modelni (`.joblib`) yuklab olish


In [ ]:
# 1-QADAM: Kerakli kutubxonalarni import qilish
import pandas as pd
import numpy as np
import re
import joblib
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, f1_score
from sklearn.base import BaseEstimator, TransformerMixin

print('✅ Barcha kutubxonalar muvaffaqiyatli yuklandi!')


In [ ]:
# 2-QADAM: Ma'lumotlar to'plamini (taxi_training_dataset.csv) yuklash
print('Iltimos, kompyuteringizdagi taxi_training_dataset.csv faylini tanlang:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# CARGO_REQUEST ni ham buyurtma sifatida PASSENGER_REQUEST ga birlashtiramiz
df['label'] = df['label'].replace({'CARGO_REQUEST': 'PASSENGER_REQUEST'})

print(f'\n🎉 Ma\'lumotlar muvaffaqiyatli yuklandi! Jami xabarlar soni: {len(df)}')
print('\n📊 Sinf bo\'yicha taqsimot:')
print(df['label'].value_counts())
df.head(5)


In [ ]:
# 3-QADAM: O'zbek tili transliteratsiyasi va xususiyatlar (Features)

CYR_TO_LAT = {
    'а': 'a', 'б': 'b', 'в': 'v', 'г': 'g', 'д': 'd', 'е': 'e', 'ё': 'yo', 'ж': 'j',
    'з': 'z', 'и': 'i', 'й': 'y', 'к': 'k', 'л': 'l', 'м': 'm', 'н': 'n', 'о': 'o',
    'п': 'p', 'р': 'r', 'с': 's', 'т': 't', 'у': 'u', 'ф': 'f', 'х': 'x', 'ц': 'ts',
    'ч': 'ch', 'ш': 'sh', 'щ': 'sh', 'ъ': "'", 'ы': 'i', 'ь': '', 'э': 'e', 'ю': 'yu',
    'я': 'ya', 'ў': "o'", 'қ': 'q', 'ғ': "g'", 'ҳ': 'h'
}

def normalize_uzbek(text: str) -> str:
    text = str(text).lower()
    text = text.replace('ў', "o'").replace('қ', 'q').replace('ғ', "g'").replace('ҳ', 'h')
    res = [CYR_TO_LAT.get(ch, ch) for ch in text]
    t = ''.join(res)
    t = re.sub(r"[^a-z0-9\s\'\`’ʻ]", ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

class UzbekPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return [f'{t} {normalize_uzbek(t)}' for t in X]

print('✅ O\'zbek tili transliterator va preprocessor tayyor!')


In [ ]:
# 4-QADAM: Modelni Train qilish (O'qitish)

X = df['text'].astype(str).tolist()
y = df['label'].tolist()

# 80% Train, 20% Test (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'O\'qitish qismi (Train): {len(X_train)} ta xabar')
print(f'Sinov qismi (Test):      {len(X_test)} ta xabar')
print('\n⏳ Model o\'qitilmoqda (Multi-gram Calibrated LinearSVC)...')

# Feature Union: So'zlar (1-2 gram) + Bo'g'in/qo'shimchalar (2-5 char n-gram)
feature_union = FeatureUnion([
    ('word_features', TfidfVectorizer(
        analyzer='word', ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=8000
    )),
    ('char_features', TfidfVectorizer(
        analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True, min_df=2, max_features=16000
    ))
])

base_svc = LinearSVC(C=1.2, class_weight='balanced', random_state=42, max_iter=3000)
calibrated_clf = CalibratedClassifierCV(estimator=base_svc, cv=3)

model = Pipeline([
    ('preprocessor', UzbekPreprocessor()),
    ('features', feature_union),
    ('clf', calibrated_clf)
])

model.fit(X_train, y_train)
print('🎉 MODEL MUVAFFAQIYATLI O\'QITILDI!')


In [ ]:
# 5-QADAM: Model imtihoni va aniqlik natijalari
y_pred = model.predict(X_test)

macro_f1 = f1_score(y_test, y_pred, average='macro')
weighted_f1 = f1_score(y_test, y_pred, average='weighted')

print('=' * 65)
print(f'🏆 UMUMIY ANIQLIK (Weighted F1): {weighted_f1 * 100:.2f}%')
print(f'📊 MACRO F1-SCORE:             {macro_f1 * 100:.2f}%')
print('=' * 65)
print('\nBatafsil hisobot:\n')
print(classification_report(y_test, y_pred, digits=4))


In [ ]:
# 6-QADAM: JONLI TEST (Playground) — Xohlagan xabaringizni yozib sinab ko'ring!

def test_xabar(text: str):
    probs = model.predict_proba([text])[0]
    classes = model.classes_
    prob_map = {cls: prob for cls, prob in zip(classes, probs)}
    pred_label = max(prob_map, key=prob_map.get)
    conf = prob_map[pred_label] * 100

    icons = {
        'PASSENGER_REQUEST': '🚕 YO\'LOVCHI / BUYURTMA (VIP guruhga chiqadi!)',
        'DRIVER_OFFER': '🚖 HAYDOVCHI TAKLIFI / REKLAMA (Bloklanadi!)',
        'OTHER': '🛑 SPAM / BEGONA XABAR (Bloklanadi!)'
    }

    print('-' * 60)
    print(f'💬 Xabar: "{text}"')
    print(f'🎯 Natija: {icons.get(pred_label, pred_label)}')
    print(f'📈 Ishonch darajasi (Confidence): {conf:.1f}%')
    print('Ehtimolliklar taqsimoti:')
    for cls_name, p in sorted(prob_map.items(), key=lambda x: -x[1]):
        bar = '█' * int(p * 20)
        print(f'  • {cls_name:<20}: {p*100:5.1f}% | {bar}')
    print('-' * 60)

# Sinov namunalari:
test_xabar("Toshkentdan Qo'qonga 2 kishi bor ketadigan")
test_xabar('Ertaga ertalabga Rishtondan Toshkonga 1 ta kam pochta olamiz kobalt')
test_xabar('Assalomu alaykum kvartira kerak 1 kishiga sheriklikka')
test_xabar('Тошкентга почта бор эди ким олиб кетади?')
test_xabar('Телеграм спам очиш хизмати арзон нархда')


In [ ]:
# 7-QADAM: Tayyor modelni (.joblib) yuklab olish
joblib.dump(model, 'taxi_classifier.joblib')
print('✅ Model saqlandi! Yuklab olish boshlanmoqda...')
files.download('taxi_classifier.joblib')
